# 19A4A — Cycle-25 Year-wise AIA Staging Preparation

## Purpose

Prepare deterministic, year-wise local staging plans for the **already-frozen independent Cycle-25 AIA test population**.

This notebook performs **storage preparation only**. It does not load the CNN-GRU model, compute predictions, fit calibration, select thresholds, or score Cycle-25.

## Frozen test contract

- Years: 2021–2025
- Targets: 49,329
- Positives: 2,351
- Regions: inherited from frozen 19A1C manifest
- Three temporal AIA frames per target: t−288, t−192, t−96 min
- Authoritative target: `label_48h_final`
- Test role: `independent_cycle25_test`

## Staging policy

Only one year is staged locally at a time:

`stage year → run frozen inference → preserve predictions → verify → delete that year's payload → stage next year`

This avoids exceeding the available local data-disk capacity and prevents unnecessary duplication.


In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

HOME = Path.home()

CANDIDATES = [
    HOME / "aia19_temporal_aia_input_corrected" / "cycle25_aia_independent_test_manifest.csv.gz",
    HOME / "aia19_temporal_aia_input_corrected" / "cycle25_aia_independent_test_manifest.csv",
]

MANIFEST = next((p for p in CANDIDATES if p.exists()), None)
if MANIFEST is None:
    raise FileNotFoundError("Cycle-25 independent AIA manifest not found.")

OUT = HOME / "aia19_cycle25_staging_prep"
URI_DIR = OUT / "uri_lists"
MAP_DIR = OUT / "target_maps"
SCRIPT_DIR = OUT / "scripts"

for d in [OUT, URI_DIR, MAP_DIR, SCRIPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = Path("/mnt/disks/aia-cache/cycle25_yearwise")

URI_COLS = [
    "history_uri_tminus288",
    "history_uri_tminus192",
    "history_uri_tminus96",
]

EXPECTED_YEAR_COUNTS = {
    2021: (5104, 126, 5516),
    2022: (9999, 119, 10913),
    2023: (12392, 278, 13566),
    2024: (12553, 1316, 13831),
    2025: (9281, 512, 10879),
}

df = pd.read_csv(MANIFEST)

print("Manifest:", MANIFEST)
print("Rows:", len(df))
print("Positives:", int(df["label_48h_final"].sum()))
print("Regions:", df["region_component_id"].nunique())

assert len(df) == 49329
assert int(df["label_48h_final"].sum()) == 2351
assert set(df["stored_year"].unique()) == set(EXPECTED_YEAR_COUNTS)


Manifest: /home/abmoses2000/aia19_temporal_aia_input_corrected/cycle25_aia_independent_test_manifest.csv.gz
Rows: 49329
Positives: 2351
Regions: 985


## 1. Build year-wise unique-object inventories

In [2]:
summary = []
all_uri_sets = {}

for year in sorted(EXPECTED_YEAR_COUNTS):
    g = df[df["stored_year"].eq(year)].copy()
    refs = pd.concat([g[c] for c in URI_COLS], ignore_index=True)
    unique_uris = pd.Index(pd.unique(refs))
    all_uri_sets[year] = set(unique_uris)

    exp_targets, exp_pos, exp_unique = EXPECTED_YEAR_COUNTS[year]
    assert len(g) == exp_targets
    assert int(g["label_48h_final"].sum()) == exp_pos
    assert len(unique_uris) == exp_unique

    # Ensure all URIs are valid GCS object paths.
    assert all(isinstance(u, str) and u.startswith("gs://") for u in unique_uris)

    # Local cache is deliberately year-scoped.
    local_dir = CACHE_ROOT / str(year)

    # Use basename because each year has a separate directory.
    inv = pd.DataFrame({"object_uri": unique_uris})
    inv["filename"] = inv["object_uri"].map(lambda u: u.rsplit("/", 1)[-1])
    inv["local_path"] = inv["filename"].map(lambda x: str(local_dir / x))

    if inv["local_path"].duplicated().any():
        dup = inv[inv["local_path"].duplicated(keep=False)]
        raise RuntimeError(f"Local-path collision for {year}:\n{dup.head()}")

    inv.to_csv(
        OUT / f"cycle25_{year}_unique_aia_objects.csv.gz",
        index=False,
        compression="gzip",
    )

    (URI_DIR / f"cycle25_{year}_uris.txt").write_text(
        "\n".join(inv["object_uri"].tolist()) + "\n"
    )

    summary.append({
        "year": year,
        "targets": len(g),
        "positives": int(g["label_48h_final"].sum()),
        "regions": int(g["region_component_id"].nunique()),
        "frame_references": len(g) * 3,
        "unique_objects": len(inv),
        "mean_reuse": (len(g) * 3) / len(inv),
        "cache_dir": str(local_dir),
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT / "cycle25_yearwise_staging_summary.csv", index=False)
print(summary_df.to_string(index=False))


 year  targets  positives  regions  frame_references  unique_objects  mean_reuse                                  cache_dir
 2021     5104        126      103             15312            5516    2.775925 /mnt/disks/aia-cache/cycle25_yearwise/2021
 2022     9999        119      190             29997           10913    2.748740 /mnt/disks/aia-cache/cycle25_yearwise/2022
 2023    12392        278      225             37176           13566    2.740380 /mnt/disks/aia-cache/cycle25_yearwise/2023
 2024    12553       1316      232             37659           13831    2.722797 /mnt/disks/aia-cache/cycle25_yearwise/2024
 2025     9281        512      240             27843           10879    2.559334 /mnt/disks/aia-cache/cycle25_yearwise/2025


## 2. Verify and document cross-year URI overlap

A small amount of overlap is valid at calendar-year boundaries because the
three-frame history of an early-January target can include late-December
objects from the previous year. We therefore document exact shared URIs and
verify the frozen global/year-wise counts rather than requiring zero overlap.


In [3]:
overlap_records = []
shared_uri_records = []

years = sorted(all_uri_sets)
for i, a in enumerate(years):
    for b in years[i+1:]:
        shared = sorted(all_uri_sets[a] & all_uri_sets[b])
        overlap_records.append({
            "year_a": a,
            "year_b": b,
            "uri_overlap": len(shared),
        })
        for uri in shared:
            shared_uri_records.append({
                "year_a": a,
                "year_b": b,
                "object_uri": uri,
            })

overlap_df = pd.DataFrame(overlap_records)
overlap_df.to_csv(OUT / "cycle25_cross_year_uri_overlap.csv", index=False)

shared_uri_df = pd.DataFrame(shared_uri_records)
if len(shared_uri_df):
    shared_uri_df.to_csv(
        OUT / "cycle25_cross_year_shared_uris.csv.gz",
        index=False,
        compression="gzip",
    )

pairwise_overlap_total = int(overlap_df["uri_overlap"].sum())
global_unique = len(set().union(*all_uri_sets.values()))
sum_yearwise_unique = sum(len(v) for v in all_uri_sets.values())
duplicate_count_from_union = sum_yearwise_unique - global_unique

print(overlap_df.to_string(index=False))
print("Pairwise overlap total:", pairwise_overlap_total)
print("Sum of year-wise unique counts:", sum_yearwise_unique)
print("Global unique objects:", global_unique)
print("Duplicate count implied by union:", duplicate_count_from_union)

assert global_unique == 54697, global_unique
assert sum_yearwise_unique == 54705, sum_yearwise_unique
assert duplicate_count_from_union == 8, duplicate_count_from_union
assert pairwise_overlap_total == 8, pairwise_overlap_total

non_adjacent = overlap_df[
    (overlap_df["uri_overlap"] > 0)
    & ((overlap_df["year_b"] - overlap_df["year_a"]) != 1)
]
assert len(non_adjacent) == 0, non_adjacent

print("\nCross-year temporal-boundary overlap check passed.")
if len(shared_uri_df):
    print("\nShared URIs:")
    print(shared_uri_df.to_string(index=False))


 year_a  year_b  uri_overlap
   2021    2022            4
   2021    2023            0
   2021    2024            0
   2021    2025            0
   2022    2023            4
   2022    2024            0
   2022    2025            0
   2023    2024            0
   2023    2025            0
   2024    2025            0
Pairwise overlap total: 8
Sum of year-wise unique counts: 54705
Global unique objects: 54697
Duplicate count implied by union: 8

Cross-year temporal-boundary overlap check passed.

Shared URIs:
 year_a  year_b                                                                                    object_uri
   2021    2022 gs://suryabench-sharp-pipeline-bamidele/samples_npz/2021/20211231_1924_HARP7896_NOAA12919.npz
   2021    2022 gs://suryabench-sharp-pipeline-bamidele/samples_npz/2021/20211231_2036_HARP7913_NOAA12922.npz
   2021    2022 gs://suryabench-sharp-pipeline-bamidele/samples_npz/2021/20211231_2100_HARP7896_NOAA12919.npz
   2021    2022 gs://suryabench-sharp-pipeline

## 3. Build year-wise target maps with deterministic local paths

In [4]:
for year in sorted(EXPECTED_YEAR_COUNTS):
    g = df[df["stored_year"].eq(year)].copy()

    local_dir = CACHE_ROOT / str(year)

    for src_col, dst_col in zip(
        URI_COLS,
        ["local_tminus288", "local_tminus192", "local_tminus96"],
    ):
        g[dst_col] = g[src_col].map(
            lambda u: str(local_dir / u.rsplit("/", 1)[-1])
        )

    out_cols = [
        "target_sample_id",
        "stored_year",
        "HARPNUM",
        "NOAA_AR_clean",
        "region_component_id",
        "label_48h_final",
        "role",
        *URI_COLS,
        "local_tminus288",
        "local_tminus192",
        "local_tminus96",
        "issue_utc",
        "issue_raw_TAI",
    ]
    out_cols = [c for c in out_cols if c in g.columns]

    g[out_cols].to_csv(
        MAP_DIR / f"cycle25_{year}_targets_local_paths.csv.gz",
        index=False,
        compression="gzip",
    )

print("Target maps written:", len(list(MAP_DIR.glob("*.csv.gz"))))


Target maps written: 5


## 4. Generate corrected staging scripts

The project previously found that `gsutil -m cp -I` did not preserve the intended one-object-per-line behavior in this workflow. The approved transfer pattern is therefore:

```bash
cat URI_LIST | gcloud storage cp --read-paths-from-stdin --no-clobber DEST/
```

Each year is downloaded into its own directory.


In [5]:
for year in sorted(EXPECTED_YEAR_COUNTS):
    uri_list = URI_DIR / f"cycle25_{year}_uris.txt"
    dest = CACHE_ROOT / str(year)
    expected = EXPECTED_YEAR_COUNTS[year][2]

    script = f'''#!/usr/bin/env bash
set -euo pipefail

YEAR={year}
LIST="{uri_list}"
DEST="{dest}"
EXPECTED={expected}

mkdir -p "$DEST"

echo "=== Stage Cycle-25 $YEAR ==="
echo "URI list: $LIST"
echo "Destination: $DEST"
echo "Expected unique objects: $EXPECTED"

cat "$LIST" | gcloud storage cp \
  --read-paths-from-stdin \
  --no-clobber \
  "$DEST/"

ACTUAL=$(find "$DEST" -maxdepth 1 -type f -name '*.npz' | wc -l)
echo "Actual local objects: $ACTUAL"

if [ "$ACTUAL" -ne "$EXPECTED" ]; then
  echo "COUNT_MISMATCH expected=$EXPECTED actual=$ACTUAL" >&2
  exit 2
fi

echo "COUNT_PASS year=$YEAR objects=$ACTUAL"
'''
    p = SCRIPT_DIR / f"stage_cycle25_{year}.sh"
    p.write_text(script)
    p.chmod(0o755)

print("Generated scripts:")
for p in sorted(SCRIPT_DIR.glob("*.sh")):
    print(p)


Generated scripts:
/home/abmoses2000/aia19_cycle25_staging_prep/scripts/stage_cycle25_2021.sh
/home/abmoses2000/aia19_cycle25_staging_prep/scripts/stage_cycle25_2022.sh
/home/abmoses2000/aia19_cycle25_staging_prep/scripts/stage_cycle25_2023.sh
/home/abmoses2000/aia19_cycle25_staging_prep/scripts/stage_cycle25_2024.sh
/home/abmoses2000/aia19_cycle25_staging_prep/scripts/stage_cycle25_2025.sh


## 5. Save protocol

In [6]:
protocol = {
    "status": "CYCLE25_YEARWISE_AIA_STAGING_PLAN_PREPARED_NO_MODEL_EVALUATION",
    "test_years": [2021, 2022, 2023, 2024, 2025],
    "targets": int(len(df)),
    "positives": int(df["label_48h_final"].sum()),
    "unique_objects_total": int(len(set().union(*all_uri_sets.values()))),
    "frame_references_total": int(len(df) * 3),
    "yearwise_policy": "stage -> frozen inference -> preserve predictions -> verify -> delete payload -> next year",
    "cache_root": str(CACHE_ROOT),
    "model_loaded": False,
    "predictions_generated": False,
    "metrics_computed": False,
    "recalibration_permitted": False,
    "threshold_reselection_permitted": False,
    "cycle25_used_for_tuning": False,
    "scientific_clearance": False,
}

(OUT / "protocol_record.json").write_text(json.dumps(protocol, indent=2) + "\n")
print(json.dumps(protocol, indent=2))
print("OUTPUT:", OUT)


{
  "status": "CYCLE25_YEARWISE_AIA_STAGING_PLAN_PREPARED_NO_MODEL_EVALUATION",
  "test_years": [
    2021,
    2022,
    2023,
    2024,
    2025
  ],
  "targets": 49329,
  "positives": 2351,
  "unique_objects_total": 54697,
  "frame_references_total": 147987,
  "yearwise_policy": "stage -> frozen inference -> preserve predictions -> verify -> delete payload -> next year",
  "cache_root": "/mnt/disks/aia-cache/cycle25_yearwise",
  "model_loaded": false,
  "predictions_generated": false,
  "metrics_computed": false,
  "recalibration_permitted": false,
  "threshold_reselection_permitted": false,
  "cycle25_used_for_tuning": false,
  "scientific_clearance": false
}
OUTPUT: /home/abmoses2000/aia19_cycle25_staging_prep


## Handoff to 19A4B

Start with 2021 only.

After download:
1. verify exact object count;
2. run byte/readability checks before inference;
3. generate predictions using the frozen 19A3 pipeline;
4. preserve predictions outside the disposable year cache;
5. only then remove the staged 2021 payload and continue to 2022.

No annual performance metrics are required during staging/inference. Final scoring occurs only after all 49,329 predictions have been concatenated and verified.
